# Notebook 15: Multi-Agent Communication Patterns (MCP/A2A)

**🎯 Goal:** Explore advanced design patterns for how multiple agents can communicate and coordinate, moving beyond a simple supervisor model.

## 🧩 Concept Overview

In a multi-agent system, the 'protocol' for how agents communicate is a critical design choice. While the centralized supervisor pattern from Notebook 13 is effective, other patterns can be more suitable for different kinds of problems.

We will explore three common patterns for Agent-to-Agent (A2A) communication:

1.  **Shared State (Hierarchical Control):** A supervisor agent directs tasks to worker agents by updating a shared state. This is a review of the pattern from Notebook 13.
2.  **Message Passing (Mailbox System):** Agents communicate by sending messages to each other's 'mailboxes'. This is great for direct, asynchronous communication.
3.  **Broadcast (Blackboard System):** An agent writes a message to a central 'blackboard' that all other agents can read. This is useful when multiple agents need to react to a single piece of information.

## 🖼️ Visual Diagrams

### Mailbox (Message Passing) Pattern
```
                     +----------------------------------+
                     | Graph State                      |
                     | Mailbox A: [Msg1]  Mailbox B: [] |
                     +----------------------------------+
                                     ^      |
                               (Write) |      | (Read)
                                     |      v
 +-----------+         Sends Msg1 to B         +-----------+
 |  Agent A  | ------------------------------> |  Agent B  |
 +-----------+                                 +-----------+
```

### Blackboard (Broadcast) Pattern
```
                               +----------------+
                               |  Blackboard    | (Central Info)
                               +-------+--------+
                                       ^      
                               (Write) |      
                                       |      
 +-----------+                 +-------+--------+       +-----------+
 |  Agent A  | -- (Reads) -->  |   Supervisor   | <---  |  Agent B  |
 +-----------+                 +----------------+       +-----------+
```

## ⚙️ Setup

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Dict
import operator
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

load_dotenv()
llm = ChatOpenAI(model="gpt-4o")

## 1. Pattern 1: Shared State (Review)

This is the hierarchical pattern from Notebook 13. A supervisor agent directly modifies the state to route tasks to specific workers. It's simple and effective for linear, predictable workflows.

## 2. Pattern 2: Message Passing / Mailbox System

Here, agents communicate by sending messages to each other. This decouples the agents, as they don't need to know about a central supervisor's logic; they only need to know how to check their mailbox and send messages.

In [ ]:
class MailboxState(TypedDict):
    # Each agent has a list of messages in their mailbox
    mailboxes: Dict[str, List[str]]
    # The next agent to run
    next_agent: str

def agent_a(state: MailboxState) -> MailboxState:
    print("---> Agent A running")
    my_messages = state['mailboxes'].get('A', [])
    print(f"Agent A received messages: {my_messages}")
    # Send a message to Agent B
    new_mailboxes = state['mailboxes'].copy()
    new_mailboxes.setdefault('B', []).append("Hello from A!")
    return {"mailboxes": new_mailboxes, "next_agent": "B"}

def agent_b(state: MailboxState) -> MailboxState:
    print("---> Agent B running")
    my_messages = state['mailboxes'].get('B', [])
    print(f"Agent B received messages: {my_messages}")
    # End the process
    return {"next_agent": "end"}

# The supervisor just checks the 'next_agent' field
def mailbox_supervisor(state: MailboxState):
    return state.get("next_agent", "A")

mailbox_graph = StateGraph(MailboxState)
mailbox_graph.add_node("A", agent_a)
mailbox_graph.add_node("B", agent_b)
mailbox_graph.set_entry_point("A")
mailbox_graph.add_conditional_edges("A", mailbox_supervisor, {"B": "B", "end": END})
mailbox_graph.add_conditional_edges("B", mailbox_supervisor, {"A": "A", "end": END})

mailbox_app = mailbox_graph.compile()

initial_mailbox_state = {"mailboxes": {"A": ["Start task"]}}
final_mailbox_state = mailbox_app.invoke(initial_mailbox_state)
print(f"\nFinal mailboxes: {final_mailbox_state['mailboxes']}")

## 3. Pattern 3: Broadcast / Blackboard System

In this pattern, there's a shared 'blackboard' that any agent can write to. After a write, a supervisor can decide which agent (or agents) should react to the new information. This is useful for more dynamic and event-driven systems.

In [ ]:
class BlackboardState(TypedDict):
    blackboard: str
    # We can use a list to track which agents have already acted
    agents_acted: List[str]

def writer_agent(state: BlackboardState) -> BlackboardState:
    print("---> Writer Agent running")
    # The writer posts a message to the blackboard
    return {"blackboard": "A new policy has been drafted.", "agents_acted": ["writer"]}

def legal_agent(state: BlackboardState) -> BlackboardState:
    print("---> Legal Agent running")
    message = state['blackboard']
    print(f"Legal agent reads: '{message}'. It looks compliant.")
    acted = state['agents_acted'] + ["legal"]
    return {"agents_acted": acted}

def finance_agent(state: BlackboardState) -> BlackboardState:
    print("---> Finance Agent running")
    message = state['blackboard']
    print(f"Finance agent reads: '{message}'. It has no budget impact.")
    acted = state['agents_acted'] + ["finance"]
    return {"agents_acted": acted}

# The supervisor decides which agent to run next
def blackboard_supervisor(state: BlackboardState):
    acted = state.get("agents_acted", [])
    if "writer" not in acted:
        return "writer"
    if "legal" not in acted:
        return "legal"
    if "finance" not in acted:
        return "finance"
    return "end"

blackboard_graph = StateGraph(BlackboardState)
blackboard_graph.add_node("writer", writer_agent)
blackboard_graph.add_node("legal", legal_agent)
blackboard_graph.add_node("finance", finance_agent)
blackboard_graph.set_entry_point("writer")
blackboard_graph.add_conditional_edges("writer", blackboard_supervisor, {"legal": "legal", "finance": "finance", "end": END})
blackboard_graph.add_conditional_edges("legal", blackboard_supervisor, {"finance": "finance", "end": END})
blackboard_graph.add_conditional_edges("finance", blackboard_supervisor, {"end": END})

blackboard_app = blackboard_graph.compile()
final_blackboard_state = blackboard_app.invoke({})
print(f"\nFinal blackboard state: {final_blackboard_state}")

## 💡 Pro Tip

Choosing the right communication pattern is a key architectural decision. There's no single best answer; it depends entirely on your use case:
- **Hierarchical Supervisor:** Best for tasks that can be broken down into a clear, linear sequence of steps. (e.g., Research -> Write -> Review).
- **Mailbox System:** Best for when agents need to send direct messages to each other, but the overall workflow is still somewhat flexible or asynchronous.
- **Blackboard System:** Best for event-driven workflows where multiple agents might need to react independently to a new piece of information.

## 🏁 Summary

You've now learned how to design the very fabric of communication within your AI teams.

1.  **Communication is Key:** The way agents pass information to each other defines how they collaborate and solve problems.
2.  **LangGraph Supports Multiple Patterns:** You can implement various communication protocols, from simple supervised workflows to more complex message-passing and broadcast systems.
3.  **Design Follows Function:** The right communication pattern depends on the nature of the task you are trying to solve. Consider the workflow before you start building the graph.